# Execute simulation

**LD-BT simulation-benchmark step.** In-silico comparison of library-construction strategies. For each strategy and colony-pick budget, reports the expected per-position mutation frequency and amino-acid diversity, averaged over random seeds.

Strategies: `eppcr` (error-prone PCR under Taq Ts:Tv bias, Cadwell & Joyce), `eppcr_theoretical` (uniform-substitution upper bound), `ssm` (site-saturation coupon-collector expected unique AAs).

Edit **Parameters**, then Run All. Output: `<output>/simulation_summary.csv`.

**Figures:** library-construction strategy comparison (Fig 6).

In [ ]:
import os, sys
ROOT = os.path.abspath(os.getcwd())
SRC = os.path.join(ROOT, 'src')
if SRC not in sys.path:
    sys.path.insert(0, SRC)
print('repo root :', ROOT)
print('src on path:', SRC in sys.path)

In [ ]:
import numpy as np
import pandas as pd
import simulations as sim
print('simulation constants loaded; COLONY_NS =', sim.COLONY_NS)

## Parameters
Defaults are the project's published simulation configuration.

In [ ]:
STRATEGIES  = ['eppcr', 'eppcr_theoretical', 'ssm']
COLONY_NS   = sim.COLONY_NS               # colony picks per round
N_SEEDS     = sim.N_SEEDS                 # seed replicates (epPCR strategies)
BASE_SEED   = sim.EPPCR_AVG_BASE_SEED
OUTPUT_PATH = os.path.join('results', 'simulation')
print('strategies :', STRATEGIES)
print('colony Ns  :', COLONY_NS)
print('n_seeds    :', N_SEEDS)

## Run the strategy sweep

In [ ]:
rows = []
for strategy in STRATEGIES:
    for rnd, N in enumerate(COLONY_NS):
        if strategy == 'ssm':
            rows.append({'strategy': strategy, 'round': rnd, 'colony_picks': N,
                         'mean_unique_aas': sim.ssm_coupon_expected(N),
                         'mean_mut_freq_pct': np.nan})
        else:
            fn = (sim.simulate_eppcr_avg if strategy == 'eppcr'
                  else sim.simulate_eppcr_theoretical_avg)
            mf, ua, _, nuniq = fn(N, n_seeds=N_SEEDS, base_seed=BASE_SEED + rnd * 1000)
            rows.append({'strategy': strategy, 'round': rnd, 'colony_picks': N,
                         'mean_unique_aas': float(np.mean(ua)),
                         'mean_mut_freq_pct': float(np.mean(mf)),
                         'mean_unique_seqs': float(nuniq)})
        print('  %-18s round %d (N=%d) done' % (strategy, rnd, N))

## Assemble and write summary

In [ ]:
summary = pd.DataFrame(rows)
os.makedirs(OUTPUT_PATH, exist_ok=True)
out = os.path.join(OUTPUT_PATH, 'simulation_summary.csv')
summary.to_csv(out, index=False)
print('written to :', out)
summary